# Max version: stacked model (about 85% accuracy)
Highest-accuracy setup found. It combines three kinds of input:

1. **116 handcrafted features**: the original 36 in `features.csv` + 80 new ones from `src/cole_features_max.py`
   (word families, story structure, emotional arc, narration style, readability, GPT-2 "AI detector" signals).
2. **GPT-2 embedding**: GPT-2 reads each transcript and summarizes it as 768 numbers (`data/gpt2_embeddings_cole.csv`).
3. **Word and character patterns**: TF-IDF of words/word pairs and of 2-5 letter chunks, learned from the text.

Each input gets its own logistic regression; a final logistic regression (**stacking**) learns how much to trust each one.

The feature and embedding files are precomputed, so this notebook only needs pandas + scikit-learn.
Regenerating them needs `torch`, `transformers` and `vaderSentiment` (see `src/cole_features_max.py`).
Running all cells takes about 5-10 minutes.

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

feats = pd.read_csv("../data/features_cole_max.csv")
emb = pd.read_csv("../data/gpt2_embeddings_cole.csv")
meta = pd.read_csv("../data/transcripts_filtered.csv")
assert (emb.video_id.values == meta.video_id.values).all()

y = feats["y"]
ORIGINAL = [c for c in pd.read_csv("../data/features.csv").columns if c not in ("transcript", "y")]
HANDCRAFTED = [c for c in feats.columns if c not in ("transcript", "y")]
EMBEDDING = [c for c in emb.columns if c != "video_id"]
X = pd.concat([feats[HANDCRAFTED], emb[EMBEDDING], meta.transcript.rename("text")], axis=1)
print(f"{len(y)} transcripts | original features: {len(ORIGINAL)} | handcrafted total: {len(HANDCRAFTED)} | embedding: {len(EMBEDDING)}")

311 transcripts | original features: 36 | handcrafted total: 116 | embedding: 768


## Models
`C` (how strongly logistic regression is regularized) is tuned with an inner 5-fold CV on the training data only.

In [2]:
inner = StratifiedKFold(5, shuffle=True, random_state=0)

def tuned_lr(cols):
    pipe = make_pipeline(ColumnTransformer([("cols", "passthrough", cols)]), StandardScaler(), LogisticRegression(max_iter=5000))
    return GridSearchCV(pipe, {"logisticregression__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1]}, cv=inner, scoring="roc_auc")

def text_lr(analyzer, ngrams, min_df):
    vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngrams, min_df=min_df, sublinear_tf=True)
    return make_pipeline(ColumnTransformer([("text", vec, "text")]), LogisticRegression(C=5, max_iter=5000))

def max_model():
    parts = [("handcrafted", tuned_lr(HANDCRAFTED)), ("gpt2_embedding", tuned_lr(EMBEDDING)),
             ("word_patterns", text_lr("word", (1, 2), 2)), ("char_patterns", text_lr("char_wb", (2, 5), 3))]
    return StackingClassifier(parts, final_estimator=LogisticRegression(max_iter=5000),
                              cv=StratifiedKFold(5, shuffle=True, random_state=1), stack_method="predict_proba")

SIMPLE_EXTRA = ["discourse_markers_per100", "casual_verbs_per100", "slang_per100", "subordinators_per100",
                "time_connectives_per100", "reveal_words_per100", "absolutist_per100", "kinship_per100",
                "future_modals_per100", "love_talk_per100", "apology_per100"]
SETUPS = {"original 36 (LogReg)": lambda: tuned_lr(ORIGINAL),
          "simple version: 36 + 11 (LogReg)": lambda: tuned_lr(ORIGINAL + SIMPLE_EXTRA),
          "max version (stacked)": max_model}

## Fair comparison on 20 train/test splits
All three setups get the **same 20 splits** (5-fold CV repeated 4 times); every choice is made on training data only.

In [3]:
splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42).split(X, y))
scores = {}
for name, make in SETUPS.items():
    acc, auc = [], []
    for tr, te in splits:
        p = make().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
        acc.append(accuracy_score(y.iloc[te], p >= 0.5)); auc.append(roc_auc_score(y.iloc[te], p))
    scores[name] = np.array(acc)
    print(f"{name:34s} accuracy {np.mean(acc):.3f} (+/- {np.std(acc):.3f})   AUC {np.mean(auc):.3f}")

base = scores["original 36 (LogReg)"]
for name in list(SETUPS)[1:]:
    print(f"{name} beats original 36 on {np.mean(scores[name] > base):.0%} of splits, loses on {np.mean(scores[name] < base):.0%}")

original 36 (LogReg)               accuracy 0.759 (+/- 0.047)   AUC 0.825
simple version: 36 + 11 (LogReg)   accuracy 0.812 (+/- 0.049)   AUC 0.873
max version (stacked)              accuracy 0.846 (+/- 0.059)   AUC 0.910
simple version: 36 + 11 (LogReg) beats original 36 on 90% of splits, loses on 10%
max version (stacked) beats original 36 on 95% of splits, loses on 0%


## Topic check: test on a theme the model has never seen
Train on 7 themes, test on the 8th, for every theme. If the model were only learning topic words
(e.g. "fruit" or "class"), accuracy would collapse here.

In [4]:
theme = meta.theme.values
for name in ["original 36 (LogReg)", "max version (stacked)"]:
    pred = np.zeros(len(y))
    for th in np.unique(theme):
        tr, te = np.where(theme != th)[0], np.where(theme == th)[0]
        pred[te] = SETUPS[name]().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    per_theme = {th: round(accuracy_score(y[theme == th], pred[theme == th] >= 0.5), 2) for th in np.unique(theme)}
    print(f"{name:24s} unseen-theme accuracy {accuracy_score(y, pred >= 0.5):.3f}  per theme: {per_theme}")

original 36 (LogReg)     unseen-theme accuracy 0.762  per theme: {'animal': 0.68, 'baby': 0.76, 'couple': 0.78, 'family': 0.81, 'fruit': 0.76, 'horror': 0.53, 'school': 0.78, 'work': 0.77}
max version (stacked)    unseen-theme accuracy 0.836  per theme: {'animal': 0.82, 'baby': 0.82, 'couple': 0.83, 'family': 0.83, 'fruit': 0.87, 'horror': 0.67, 'school': 0.84, 'work': 0.93}


## Classification report (max version, 5-fold out-of-fold predictions)

In [5]:
oof = cross_val_predict(max_model(), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]
print(classification_report(y, oof >= 0.5, target_names=["human", "AI"]))

              precision    recall  f1-score   support

       human       0.83      0.89      0.86       158
          AI       0.88      0.81      0.84       153

    accuracy                           0.85       311
   macro avg       0.85      0.85      0.85       311
weighted avg       0.85      0.85      0.85       311



## How much the final model trusts each input
Weights of the final (stacking) logistic regression, fitted on all clips: one weight per input model.

In [6]:
final = max_model().fit(X, y)
weights = pd.Series(final.final_estimator_.coef_[0], index=[name for name, _ in final.estimators])
print(weights.round(2).to_string())

handcrafted       2.82
gpt2_embedding    2.08
word_patterns     2.05
char_patterns     1.82
